In [ ]:
class BatchNormNet(nn.Module):
    def __init__(self, hidden_dim, out_dim, device, dtype, drop_out=0.1):
        super(BatchNormNet, self).__init__()
        self.hidden_dim1 = hidden_dim
        self.out_dim = out_dim
        self.drop_out = drop_out
        self.device = device
        self.dtype = dtype

        self.fc1 = nn.Linear(1024, hidden_dim, dtype=dtype, device=device)
        self.bn1 = nn.BatchNorm1d(hidden_dim, dtype=dtype, device=device)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim,
                             dtype=dtype, device=device)
        self.bn2 = nn.BatchNorm1d(hidden_dim, dtype=dtype, device=device)
        self.fc3 = nn.Linear(hidden_dim, out_dim, dtype=dtype, device=device)
        self.dropout = nn.Dropout(p=drop_out)

    def forward(self, x):
        x = self.dropout(self.bn1(self.fc1(x)))
        x = torch.relu(x)
        x = self.dropout(self.bn2(self.fc2(x)))
        x = torch.relu(x)
        x = self.fc3(x)
        return x

In [ ]:
from torchvision.models import resnet18
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, common_dim, salient_dim):
        super(Encoder, self).__init__()

        self.common_dim = 512
        self.salient_dim = 512

        
        # Add MLP projection.
        self.common_projector = nn.Sequential(nn.Linear(1024, 512),
                                       nn.BatchNorm1d(512),
                                       nn.Linear(512, 256),
                                       nn.BatchNorm1d(256),
                                              
                                       nn.ReLU(),
                                       nn.Linear(256, common_dim))

        self.specific_projector = nn.Sequential(nn.Linear(1024, 512),
                                       nn.BatchNorm1d(512),
                                       nn.Linear(512, 256),
                                       nn.BatchNorm1d(256),
                                              
                                       nn.ReLU(),
                                              nn.Linear(256, salient_dim))

    def forward(self, x):
        #common_rep = self.common_enc(x)
        #specific_rep = self.specific_enc(x)

        common_head = self.common_projector(x)
        specific_head = self.specific_projector(x)

        return  common_head,  specific_head

In [ ]:
file1 = '../local_snapshot/data/Embedding/split100_prostt5.h5'
file2 = '../local_snapshot/data/Embedding/Bag_100.h5'
file_label = '../local_snapshot/data/Disen_Data_Labels_int.h5'

In [ ]:
import h5py
emb_dic = {}
with h5py.File(file1 , 'r') as hf1:
        for key in hf1.keys():
            emb_dic[key]=hf1[key]
with h5py.File(file2 , 'r') as hf2:
        for key in hf2.keys():
            emb_dic[key]=hf2[key]

In [ ]:
emb_label = {}
with h5py.File(file_label, 'r') as hf:
    for key in hf.keys():
        emb_label[key]=hf[key]

In [ ]:

def load_prost5(lookup):
    if lookup not in emb_dic.keys():
        dataset =  emb_dic[lookup.split('_')[0]]
        data = dataset[:]
    else:
        dataset =  emb_dic[lookup]
        data = dataset[:]

    torch_data = torch.tensor(data)  
    return torch_data.unsqueeze(0)

In [ ]:
import numpy as np
import torch
from keras.datasets import cifar10
from torch.utils.data import Dataset
import tensorflow.compat.v2 as tf

class CifarMNISTDataset(Dataset):
    def __init__(self, data, cifar_targets, digit_targets, background_targets):
        self.data = torch.from_numpy(np.transpose(data, (0, 3, 1, 2)).astype(float)).float()
        self.cifar_targets = torch.from_numpy(cifar_targets.astype(float)).float()
        self.digit_targets = torch.from_numpy(digit_targets.astype(float)).float()
        self.background_targets = torch.from_numpy(background_targets.astype(float)).float()

    def __len__(self):
        return len(self.cifar_targets)

    def __getitem__(self, index: int):
        """
        Args:
            index (int): Index

        Returns:
            tuple: (img, cifar_labels, digit_label, background_target_label).
        """
        img, cifar_target, digit_target, background_target_label = self.data[index], int(self.cifar_targets[index]), int(self.digit_targets[index]), int(self.background_targets[index])

        return img, cifar_target, digit_target, background_target_label

def get_cifar_mnist_datasets():
    
    X_train = []
    y_train = []
    y_train_mnist = []
    y_train_cifar = []
    for key in emb_label.keys():
        X_train.append(load_prost5(key))
        y_train_cifar.append(emb_label[key][0])
        y_train_mnist.append(emb_label[key][1])
        y_train.append(emb_label[key][2])
        
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train_mnist = np.array(y_train_mnist)
    y_train_cifar = np.array(y_train_cifar)

    

    # Instantiate MNIST superimposed on CIFAR-10 datasets
    train_dataset = CifarMNISTDataset(X_train, y_train_cifar, y_train_mnist, y_train)
   

    return train_dataset, test_dataset

In [ ]:
from torch.optim import Adam
import torch

from architectures import Encoder
from data_augmentation import CifarMNISTDataAugmentation
from dataset import get_cifar_mnist_datasets
from evaluation_script import evaluate_representations
from losses import align_loss, uniform_loss, joint_entropy_loss

# ignore warning
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning
simplefilter("ignore", category=ConvergenceWarning)

import random
random.seed(0)

# instantiate the hyper-parameters
batch_size = 512
n_epochs = 251
common_size=32
salient_size=32
salient_reg_beta = 1000
salient_term_beta = 1000
jem_lambda = 10

def norm(x):
    return torch.nn.functional.normalize(x, dim=1, p=2)

def train(encoder, train_loader):
    encoder.train()
    train_loss = 0
    for batch_idx, (view, _, _, labels) in enumerate(train_loader):
        with torch.no_grad():
            view1 = data_aug_pipeline(view)
            view2 = data_aug_pipeline(view)
            labels = labels.cuda()

        # compute common and salient spaces
        _, common_head_1, _, specific_head_1 = encoder(view1.cuda())
        _, common_head_2, _, specific_head_2 = encoder(view2.cuda())

        # hard code the fact that f_{\theta_S}(x) = s'
        specific_head_1_teacher = torch.clone(specific_head_1)
        specific_head_2_teacher = torch.clone(specific_head_2)
        specific_head_1_teacher[labels==0, :] = 0.0 # s' = 0.0, i.e: the null vector
        specific_head_2_teacher[labels==0, :] = 0.0 # s' = 0.0, i.e: the null vector

        # common term
        cm_align_loss = align_loss(norm(common_head_1), norm(common_head_2))
        cm_uniform_loss = (uniform_loss(norm(common_head_2)) + uniform_loss(norm(common_head_1))) / 2.0
        cm_loss = cm_align_loss + 2.0 * cm_uniform_loss

        # salient term
        sp_align_loss = align_loss(specific_head_1[labels==1], specific_head_2[labels==1])
        sp_uniform_loss = (uniform_loss(specific_head_2_teacher) + uniform_loss(specific_head_1_teacher)) / 2.0
        sp_loss = sp_align_loss + 2.0 * sp_uniform_loss

        # information-less constraint
        sp_reg_loss = (align_loss(specific_head_1_teacher[labels==0], specific_head_1[labels==0]) +
                         align_loss(specific_head_2_teacher[labels==0], specific_head_2[labels==0])) / 2.0

        # null mutual information constraint
        jem_loss = joint_entropy_loss(norm(common_head_1), specific_head_1_teacher)
        jem_loss = jem_loss + joint_entropy_loss(norm(common_head_2), specific_head_2_teacher)
        jem_loss = jem_loss / 2.0

        # aggregate the losses
        loss = cm_loss + salient_term_beta*sp_loss + salient_reg_beta*sp_reg_loss + jem_lambda*jem_loss

        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        optimizer.zero_grad()

# get datasets and data loaders
train_dataset, test_dataset = get_cifar_mnist_datasets()
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size)

# instantiate Data Augmentation pipeline
data_aug_pipeline = CifarMNISTDataAugmentation()

# instantiate the contrastive encoder
encoder = Encoder(common_dim=common_size, salient_dim=salient_size).float().cuda()

# instantiate the optimizer
optimizer = Adam(list(encoder.parameters()), lr=3e-4)

# train and evaluate the method
for epoch in range(1, n_epochs):
    train(encoder, train_loader)
    if epoch % 50 == 0:
        evaluate_representations(encoder, train_loader, test_loader)
        print("---")
evaluate_representations(encoder, train_loader, test_loader)

torch.save(encoder, "cifar-mnist_k-jem_sep_clr.pth")
